# Tutorial 3 — Generative retrieval: writing the next item

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juanmigutierrez/generative-recommendation-engine/blob/main/notebooks/tutorial_03_generative_retrieval.ipynb)

Companion to the *Generative retrieval* section. Here you will:

1. turn a real user's history into the token stream the model reads,
2. **train the Semantic-ID Transformer on a subset** for a few minutes (or load the fully trained checkpoint — both paths are here),
3. run constrained beam search step by step and watch the model pick a *family* of items with the first digit before pinning one down,
4. compare its top-10 against the popularity list for any user, with the true next item marked.

Use a GPU runtime. The project's own JAX code is used throughout (`backend/models/transformer.py`, `backend/scripts/evaluate_retrieval.py`).

In [ ]:
#@title Setup — clone the repo, install deps, download the pre-computed artifacts (~1 min)
import os, sys, urllib.request
REPO_URL = "https://github.com/juanmigutierrez/generative-recommendation-engine"
RELEASE  = REPO_URL + "/releases/download/v1.0-artifacts"
QUICK    = os.environ.get("TUTORIAL_QUICK") == "1"   # tiny sizes for headless smoke tests

if not os.path.exists("backend"):
    if not os.path.exists("generative-recommendation-engine"):
        !git clone -q {REPO_URL}
    %cd generative-recommendation-engine
if "google.colab" in sys.modules:
    !pip install -q implicit lightgbm pyarrow ipywidgets 2>&1 | tail -1
os.makedirs("data/processed", exist_ok=True)

def fetch(*names):
    """Download an artifact from the GitHub release unless it is already on disk."""
    for n in names:
        p = os.path.join("data", "processed", n)
        if not os.path.exists(p):
            print("downloading", n, "...")
            urllib.request.urlretrieve(f"{RELEASE}/{n}", p)

fetch("item_catalog.parquet", "item_tokens.parquet", "loo_train_sequences.parquet", "loo_val_targets.parquet", "transformer_train_sequences_loo.npz", "transformer_vocab_meta_loo.json", "transformer_checkpoint_loo.pkl")
for p in ["backend", "backend/scripts"]:
    if p not in sys.path: sys.path.insert(0, p)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets
pd.set_option("display.max_colwidth", 90)
P = os.path.join("data", "processed")
print("ready")

## 1. Step 1: the history becomes a sentence

Each item is its 4-digit Semantic ID (three levels + tie-break), offset into one shared vocabulary; a hashed user token goes in front. This is the leave-one-out training data (`build_semantic_sequences.py --loo`): the last two items of every user are held out for evaluation.

In [ ]:
import json, pickle, jax, jax.numpy as jnp, time
from models import transformer as tx
import evaluate_retrieval as ev

items = pd.read_parquet(f"{P}/item_catalog.parquet").set_index("item_id"); title = items["description"]
item_tokens = pd.read_parquet(f"{P}/item_tokens.parquet"); item_to_tok = {r.item_id: [r.tok1, r.tok2, r.tok3, r.tok4] for r in item_tokens.itertuples(index=False)}
meta = json.load(open(f"{P}/transformer_vocab_meta_loo.json"))
seqs = {r.user_id: list(r.item_ids) for r in pd.read_parquet(f"{P}/loo_train_sequences.parquet").itertuples()}
val_target = {r.user_id: r.item_ids[0] for r in pd.read_parquet(f"{P}/loo_val_targets.parquet").itertuples()}
data = np.load(f"{P}/transformer_train_sequences_loo.npz"); tokens_all, user_ids_all = data["tokens"], data["user_ids"]
print("vocab", meta["vocab_size"], "| pad", meta["pad_token"], "| max tokens", meta["max_len"], "| users", tokens_all.shape[0])

def user_token(u): return meta["item_vocab"] + int(u) % meta["user_buckets"]
def context_tokens(u, max_items=meta["max_items"] - 1):
    return [user_token(u)] + sum((item_to_tok[i] for i in seqs[u][-max_items:]), [])

rng = np.random.RandomState(1)
demo_users = sorted(rng.choice([u for u, s in seqs.items() if 4 <= len(s) <= 8], 40, replace=False).tolist())

def show_stream(user_id):
    print(f"user {user_id} — history ({len(seqs[user_id])} items):")
    for i in seqs[user_id]: print(f"   {tuple(item_to_tok[i])}  {title[i][:75]}")
    print("\ntoken stream the model reads:", context_tokens(user_id))
    print("\nheld-out next item (val target):", title[val_target[user_id]][:80], tuple(item_to_tok[val_target[user_id]]))

interact(show_stream, user_id=widgets.Dropdown(options=demo_users, description="user"));

## 2. Step 2: train it — or load it

**Option A** trains a small model on a subset of users right here (a few minutes on a T4; the loss will still be high). **Option B** loads the checkpoint from the post (4 layers, d=128, 45 epochs on the full data). The rest of the notebook works with whichever you ran last.

In [ ]:
#@title Option A — train a small model on a subset (GPU: ~3 min)
TRAIN_HERE  = True      #@param {type:"boolean"}
N_USERS     = 20000     #@param {type:"integer"}
EPOCHS      = 6         #@param {type:"integer"}
D_MODEL, N_HEADS, N_LAYERS, D_FF, BATCH, LR, DROPOUT = 64, 4, 2, 256, 256, 1e-3, 0.1
if QUICK: N_USERS, EPOCHS = 2000, 1

if TRAIN_HERE:
    x = jnp.array(tokens_all[:N_USERS]); n = x.shape[0]
    params = tx.init_params(jax.random.PRNGKey(0), meta["vocab_size"], meta["max_len"], D_MODEL, N_HEADS, N_LAYERS, D_FF)
    opt_init, opt_update = tx.make_adam(lr=LR); opt_state = opt_init(params)
    step = jax.jit(jax.value_and_grad(lambda p, b, k: tx.loss_fn(p, b, meta["pad_token"], N_HEADS, DROPOUT, k)))
    key = jax.random.PRNGKey(1); prng = np.random.RandomState(0); hist = []; t0 = time.time()
    for ep in range(EPOCHS):
        perm = prng.permutation(n); tot = 0.0; nb_ = n // BATCH
        for b in range(nb_):
            key, sub = jax.random.split(key)
            loss, g = step(params, x[perm[b*BATCH:(b+1)*BATCH]], sub); params, opt_state = opt_update(g, opt_state, params); tot += float(loss)
        hist.append(tot / nb_); print(f"epoch {ep+1}  loss/token {hist[-1]:.3f}  ({time.time()-t0:.0f}s)")
    model_params, model_heads, model_name = params, N_HEADS, f"small model trained here ({N_USERS:,} users, {EPOCHS} epochs)"
    print("random-guess loss would be ln(vocab) =", round(float(np.log(meta["vocab_size"])), 2))

In [ ]:
#@title Option B — load the fully trained checkpoint from the post
LOAD_CHECKPOINT = True  #@param {type:"boolean"}
if LOAD_CHECKPOINT:
    raw = pickle.load(open(f"{P}/transformer_checkpoint_loo.pkl", "rb"))
    model_params = jax.tree_util.tree_map(jnp.array, raw["params"]); model_heads = raw["config"]["n_heads"]
    model_name = f"checkpoint from the post ({raw['config']['n_layers']} layers, d={raw['config']['d_model']}, epoch {raw['epoch']}, loss {raw['history'][-1]:.2f})"
    fig, ax = plt.subplots(figsize=(6, 3)); ax.plot(raw["history"], color="#2a78d6"); ax.set_xlabel("epoch"); ax.set_ylabel("loss per digit"); ax.set_title("training curve of the post's model")
    for s in ["top", "right"]: ax.spines[s].set_visible(False)
    plt.show()
print("active model:", model_name)

## 3. Steps 3–4: generate, constrained to real items

The trie below is built from every catalog item's 4 digits. At each step the model only scores digits that can still complete a real item. Pick a user: you see the *families* the first digit points at, then the finished top-10 with the true next item marked.

In [ ]:
trie = ev.build_trie(item_tokens); ev.N_HEADS = model_heads; ev.BEAM_WIDTH = 30
level1_members = item_tokens.groupby("tok1")["item_id"].apply(list).to_dict()

def generate(user_id, K=10):
    ctx = context_tokens(user_id); seen = set(seqs[user_id]); truth = val_target[user_id]
    # digit 1: what families does the model reach for?
    logits = np.array(ev.get_batch_logits(model_params, np.array([ctx + [meta["pad_token"]] * (meta["max_len"] - len(ctx))], dtype=np.int32), meta["pad_token"]))[0]
    lp = logits - np.logaddexp.reduce(logits); valid = np.array(sorted(trie)); top1 = valid[np.argsort(-lp[valid])[:3]]
    print(f"history: {' | '.join(title[i][:35] for i in seqs[user_id][-4:])}\n")
    print("first digit — the three most likely families and a sample of what lives in each:")
    for t in top1:
        members = level1_members[int(t)]
        print(f"  digit {int(t):3d}  p={np.exp(lp[t]):.2f}  ({len(members):,} items)  e.g. " + " / ".join(title[i][:28] for i in rng.choice(members, 3)))
    # full beam search
    ranked = ev.beam_search_batch(model_params, [ctx], meta, trie)[0]
    recs = [(i, s) for i, s in ranked if i not in seen][:K]
    print(f"\ntop-{K} after constrained beam search (✔ = the item the user actually reviewed next):")
    for r, (i, s) in enumerate(recs, 1):
        print(f"  {'✔' if i == truth else ' '} {r:>2}. p={np.exp(s):.4f}  {title[i][:75]}")
    if truth not in [i for i, _ in recs]: print(f"\n  (true next item: {title[truth][:75]})")

interact(generate, user_id=widgets.Dropdown(options=demo_users, description="user"), K=widgets.IntSlider(10, 5, 20, 5));

### Find the Nintendo Switch user from the post

Any user whose recent history is all one platform makes the point: the model is never told the platform.

In [ ]:
KEYWORD = "Switch"  #@param {type:"string"}
def platform_users(word, min_items=4):
    out = []
    for u, s in seqs.items():
        last = s[-min_items:]
        if len(last) == min_items and all(word.lower() in title[i].lower() for i in last): out.append(u)
    return out
pu = platform_users(KEYWORD)
print(f"{len(pu)} users whose last 4 items all mention '{KEYWORD}'")
if pu: generate(pu[0])

## 4. How much better than popularity? (small sample)

Recall@10 on a few hundred held-out val targets, versus the popularity list. Tutorial 5 does this properly on all 94,762 users with SASRec in the mix.

In [ ]:
from models.metrics import recall_at_k, ndcg_at_k
N_EVAL = 100 if QUICK else 500
train_rows = pd.DataFrame({"user_id": np.repeat(list(seqs), [len(s) for s in seqs.values()]), "item_id": np.concatenate(list(seqs.values()))})
pop_rank = train_rows.groupby("item_id").size().sort_values(ascending=False).index.tolist()
eval_users = rng.choice(sorted(seqs), N_EVAL, replace=False)
ctxs = [context_tokens(u) for u in eval_users]; r_tf, r_pop, n_tf = [], [], []
for s in range(0, N_EVAL, 50):
    for u, ranked in zip(eval_users[s:s+50], ev.beam_search_batch(model_params, ctxs[s:s+50], meta, trie)):
        seen = set(seqs[u]); recs = [i for i, _ in ranked if i not in seen][:10]; truth = {val_target[u]}
        r_tf.append(recall_at_k(recs, truth, 10)); n_tf.append(ndcg_at_k(recs, truth, 10))
        r_pop.append(recall_at_k([i for i in pop_rank if i not in seen][:10], truth, 10))
print(f"{model_name}\n  recall@10  transformer {np.mean(r_tf):.3f}   popularity {np.mean(r_pop):.3f}   (ndcg@10 transformer {np.mean(n_tf):.3f}; {N_EVAL} users)")

**Next:** [Tutorial 4 — Ranking](https://colab.research.google.com/github/juanmigutierrez/generative-recommendation-engine/blob/main/notebooks/tutorial_04_ranking.ipynb): a second opinion on the shortlist — including the bug that made the first reranker learn the opposite of what it should.